# Gemma 4 31B V9.0: True Sparse-Active 31B (`10/84` MLP + `10/32` Attn) + Outlier-Dispersion SpinQuant + End-to-End QAT

**Target:** Single `<= 5.6 GiB` Pi 4 RAM-resident brain using fast-kernel NEON (`46 GMAC/s` `1b_asym_RM_COLBLK` + `33 GMAC/s` `2b_gauss`, zero Ternary-5) running EAGLE-3 Tree Speculative Verification + `kc_step=24` Draft-Seeded Cauchy-Schwarz LM-Head (`0 ULP` loss) to reach **`20–50 tok/s`** and **`>= 100%`** stock Gemma 4 31B generalized intelligence.
- **Global Outlier-Dispersion Permutation (`Pi in S_5376`):** Exact lossless channel permutation across embeddings, all 60 layers, Vision projector, and Assistant (0 approximation error on Gemma 4 per-channel norms).
- **Cortical Column Co-Activation (`10/84 = 11.9%` MLP) + 32-Head Attention Router (`10/32 = 31.25%` Q/O on `L2..L57`):** Aligned `1b_asym_RM_COLBLOCK` (`0x61`) + `2b_gauss` (`0xA2`) streaming.
- **Sparse-Route Scale/Norm/C8DB QAT + 2-Step Rollout 4-Layer EAGLE-3 `Assistant-V9` (`Q2`/`Q4` + `BHDR-384`).**

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/DannyB-bit/Language-U-Gemma4.git"
REPO_DIR = Path("/content/Language-U-Gemma4")
DRIVE_V90 = Path("/content/drive/MyDrive/gemma4_v9_0_sparse_qat")
DRIVE_V90.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-b", "v5-qat-dev", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "v5-qat-dev"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/v5-qat-dev"], check=True)

print(subprocess.check_output(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], text=True).strip(), flush=True)
print(subprocess.check_output(["nvidia-smi"], text=True), flush=True)

import torch
assert torch.cuda.is_available(), "CUDA A100/G4/L4 GPU required!"
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[GPU Ready] {torch.cuda.get_device_name(0)} | VRAM={vram_gb:.2f} GiB", flush=True)

_need_pip = False
try:
    import datasets, safetensors, tokenizers, huggingface_hub, PIL
    from transformers.models.gemma4.modeling_gemma4 import Gemma4VisionModel
except Exception:
    _need_pip = True
if _need_pip:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "transformers>=4.51.0", "accelerate", "safetensors", "huggingface_hub", "datasets", "tokenizers", "pillow"],
        check=True,
    )

try:
    from google.colab import userdata
    hf_tok = userdata.get("HF_TOKEN")
    if hf_tok:
        os.environ["HF_TOKEN"] = hf_tok
except Exception:
    pass

SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v9_0_sparse_spinquant_qat_gpu.py"
print(subprocess.check_output([sys.executable, str(SCRIPT), "--selftest"], text=True), flush=True)

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

REPO_DIR = Path("/content/Language-U-Gemma4")
SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v9_0_sparse_spinquant_qat_gpu.py"
LOG_PATH = Path("/content/v9_0_train.log")
DRIVE_V90 = Path("/content/drive/MyDrive/gemma4_v9_0_sparse_qat")

env = os.environ.copy()
env["V9_0_DRIVE_DIR"] = str(DRIVE_V90)
env["V9_0_OUT_DIR"] = "/content/v9_0_outputs"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"
env["COLAB_AUTO_UNASSIGN"] = "1"
env["PYTHONUNBUFFERED"] = "1"

with open(LOG_PATH, "w", encoding="utf-8") as lfp:
    proc = subprocess.Popen([sys.executable, "-u", str(SCRIPT), "--run"], stdout=lfp, stderr=subprocess.STDOUT, env=env)

seen = 0
while proc.poll() is None:
    time.sleep(15)
    if LOG_PATH.is_file():
        txt = LOG_PATH.read_text(encoding="utf-8", errors="replace")
        if len(txt) > seen:
            print(txt[seen:], end="", flush=True)
            seen = len(txt)
txt = LOG_PATH.read_text(encoding="utf-8", errors="replace")
if len(txt) > seen:
    print(txt[seen:], end="", flush=True)
assert proc.returncode == 0, f"V9.0 pipeline exited with code {proc.returncode}"
print((DRIVE_V90 / "v9_0_report.json").read_text(encoding="utf-8"))